# ace-kernels Sprint 2 on Kaggle GPU

Settings (right panel): **Accelerator: GPU T4 x1**, **Internet: on**.

Add-ons -> Secrets (attach to this notebook):
- `GITHUB_TOKEN`: only if the repo is private (fine-grained, read-only, this repo only)
- `WANDB_API_KEY`: optional, mirrors results to Weights & Biases

Set `REPO` below, then *Run all*. Results: `results.tgz` in the output panel.

In [ ]:
REPO = "github.com/<your-user>/loss_optimize"  # <- edit
BRANCH = "main"

import os

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ("GITHUB_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[name] = secrets.get_secret(name)  # never printed
        print(f"{name}: set")
    except Exception:
        print(f"{name}: not attached")

In [ ]:
import subprocess
from pathlib import Path

workdir = Path("/kaggle/working/loss_optimize")
token = os.environ.get("GITHUB_TOKEN")
url = f"https://x-access-token:{token}@{REPO}.git" if token else f"https://{REPO}.git"
if not workdir.exists():
    subprocess.run(
        ["git", "clone", "-q", "--branch", BRANCH, url, str(workdir)], check=True
    )
    # drop the token from the stored remote URL
    subprocess.run(
        [
            "git",
            "-C",
            str(workdir),
            "remote",
            "set-url",
            "origin",
            f"https://{REPO}.git",
        ],
        check=True,
    )
os.chdir(workdir)
print(
    subprocess.run(
        ["git", "log", "--oneline", "-1"], capture_output=True, text=True
    ).stdout
)

In [ ]:
!bash kaggle/run_sprint2.sh 2>&1 | grep -v USDT

In [ ]:
import shutil

shutil.copy("results.tgz", "/kaggle/working/results.tgz")
print("download /kaggle/working/results.tgz from the Output panel")